<a href="https://colab.research.google.com/github/komazawa-deep-learning/komazawa-deep-learning.github.io/blob/master/2026notebooks/2026ai_lect06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 人工知能I 第6回：教師なし学習と次元削減
## オリベッティ顔データベース：PCA理論・NMF・k-means

**担当**: 浅川伸一 | **2026年度 前期**

---

## 本日の構成

| セクション | 内容 | 実習 |
|-----------|------|------|
| 1 | データ準備（第5回の引き継ぎ） | - |
| 2 | PCA の理論的理解 | 固有値・SVD・情報量 |
| 3 | NMF（非負値行列因子分解） | 固有顔 vs 局所顔 |
| 4 | k-means クラスタリング | 顔のグループ化 |
| 5 | SOM（概観・可視化） | トポロジーマップ |
| 6 | TF-IDF / Naive Bayes（概説） | デモのみ |

**第5回との接続**: PCA を「使った」→ 今日は「理解する」

---
## 0. 環境準備

In [2]:
# 日本語フォント設定（最初に実行）
!pip install japanize-matplotlib minisom --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
fm.fontManager.__init__()
import japanize_matplotlib

import seaborn as sns

from sklearn.datasets import fetch_olivetti_faces
from sklearn.decomposition import PCA, NMF, TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import normalize

import warnings
# warnings.filterwarnings('ignore')
np.random.seed(42)
# plt.style.use('seaborn-v0_8-darkgrid')
print('環境準備完了')

環境準備完了


---
## 1. データ準備（第5回の引き継ぎ）

第5回と同じオリベッティ顔データセットを使用します。
今回は男女ラベルも継続使用しますが，教師なし学習（NMF・k-means）では**ラベルを使いません**。

In [ ]:
# オリベッティ顔データセット
data = fetch_olivetti_faces()
X, y = data.data, data.target  # X: (400, 4096), y: 人物ID（0-39）

# 男女ラベル（第5回と同じ）
y_sex = np.zeros_like(y)
for woman_start in [70, 310, 340]:
    for i in range(10):
        y_sex[woman_start + i] = 1

print('=== データ概要 ===')
print(f'  サンプル数 : {X.shape[0]}')
print(f'  特徴量数  : {X.shape[1]}（64×64 ピクセル）')
print(f'  人物ID数  : {len(np.unique(y))} 人')
print(f'  男性      : {(y_sex==0).sum()} 枚')
print(f'  女性      : {(y_sex==1).sum()} 枚')

# 平均顔の可視化
mean_face = X.mean(axis=0).reshape(64, 64)
plt.figure(figsize=(3, 3))
plt.imshow(mean_face, cmap='gray')
plt.title('平均顔', fontsize=12)
plt.axis('off')
plt.tight_layout()
plt.show()

# 訓練・テスト分割（第5回と同じ設定）
X_train, X_test, y_train, y_test, ys_train, ys_test = train_test_split(
    X, y, y_sex, test_size=0.25, stratify=y_sex, random_state=42
)

---
## 2. PCA の理論的理解

**第5回では**: `PCA(n_components=100).fit_transform(X)` を使った（ブラックボックス）  
**今日は**: なぜ機能するのかを理解する

### 2.1 固有値と寄与率

In [ ]:
# 全成分での PCA（n_components 指定なし）
pca_full = PCA(random_state=42)
pca_full.fit(X_train)

explained = pca_full.explained_variance_ratio_
cumulative = np.cumsum(explained)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 左：各主成分の寄与率
axes[0].bar(range(1, 51), explained[:50], color='steelblue', alpha=0.8)
axes[0].set_xlabel('主成分の番号', fontsize=11)
axes[0].set_ylabel('寄与率', fontsize=11)
axes[0].set_title('各主成分の寄与率（上位50成分）\n第1主成分が最も大きな分散を説明', fontsize=11)

# 右：累積寄与率
axes[1].plot(range(1, len(cumulative)+1), cumulative, 'steelblue', linewidth=2)
for threshold, color, label in [(0.80,'tomato','80%'), (0.90,'green','90%'), (0.95,'orange','95%')]:
    n = np.argmax(cumulative >= threshold) + 1
    axes[1].axhline(threshold, color=color, linestyle='--', linewidth=1, alpha=0.7)
    axes[1].axvline(n, color=color, linestyle='--', linewidth=1, alpha=0.7)
    axes[1].text(n+2, threshold-0.02, f'{label}: {n}成分', fontsize=9, color=color)
axes[1].set_xlabel('主成分の数', fontsize=11)
axes[1].set_ylabel('累積寄与率', fontsize=11)
axes[1].set_title('累積寄与率\n第5回で n_components=100 を選んだ根拠', fontsize=11)
axes[1].set_xlim(0, 250)

plt.tight_layout()
plt.show()

for thr in [0.80, 0.90, 0.95]:
    n = np.argmax(cumulative >= thr) + 1
    print(f'  {thr*100:.0f}% を説明するのに必要な主成分数: {n}')

### 2.2 固有顔の理解：主成分 = 顔の「変動の方向」

In [ ]:
# 固有顔の可視化（上位20成分）
n_show = 20
fig, axes = plt.subplots(2, 10, figsize=(18, 4))

for i, ax in enumerate(axes.ravel()):
    eigenface = pca_full.components_[i].reshape(64, 64)
    # 正規化して表示（正負両方の成分があるためRdBuカラーマップ）
    vmax = np.abs(eigenface).max()
    ax.imshow(eigenface, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    var = pca_full.explained_variance_ratio_[i] * 100
    ax.set_title(f'PC{i+1}\n{var:.1f}%', fontsize=7)
    ax.axis('off')

plt.suptitle(
    '固有顔（上位20主成分）\n'
    '赤=正の成分, 青=負の成分 → 顔全体にわたるパターン（局所的ではない）',
    fontsize=11
)
plt.tight_layout()
plt.show()

In [ ]:
# 主成分数を変えて再構成品質を確認
sample_idx = 0
sample_face = X_test[sample_idx]

n_components_list = [1, 5, 10, 30, 50, 100, 150, 200]
fig, axes = plt.subplots(1, len(n_components_list)+1, figsize=(18, 3))

axes[0].imshow(sample_face.reshape(64,64), cmap='gray')
axes[0].set_title('元画像', fontsize=10)
axes[0].axis('off')

for ax, n in zip(axes[1:], n_components_list):
    pca_n = PCA(n_components=n, random_state=42).fit(X_train)
    projected   = pca_n.transform(sample_face.reshape(1,-1))
    reconstructed = pca_n.inverse_transform(projected).reshape(64,64)
    mse = np.mean((sample_face - reconstructed.ravel())**2)
    ax.imshow(reconstructed, cmap='gray')
    ax.set_title(f'n={n}\nMSE={mse:.4f}', fontsize=9)
    ax.axis('off')

plt.suptitle('主成分数と再構成品質\n少ない主成分でも顔の特徴は保存される', fontsize=11)
plt.tight_layout()
plt.show()

### 2.3 PCA と SVD の関係

In [ ]:
# SVD と PCA が同じ結果を与えることを確認
from sklearn.decomposition import TruncatedSVD

n_comp = 10

# PCA の主成分
pca_10 = PCA(n_components=n_comp, random_state=42).fit(X_train)
pca_components = pca_10.components_

# SVD の右特異ベクトル（中心化してから適用）
X_centered = X_train - X_train.mean(axis=0)
U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
svd_components = Vt[:n_comp]

# 一致の確認（符号の反転は許容）
correlations = [abs(np.dot(pca_components[i], svd_components[i]))
                for i in range(n_comp)]

print('=== PCA主成分 と SVD右特異ベクトルの一致確認 ===')
print('内積の絶対値（1.0 = 完全一致）')
for i, c in enumerate(correlations):
    print(f'  PC{i+1}: {c:.6f}')

print('\n→ PCAはSVDで実装されている（sklearn内部も同様）')
print('  固有値 = 特異値² / (n-1)')

---
## 3. NMF（非負値行列因子分解）

$$V \approx W \times H \quad (V, W, H \geq 0)$$

- $V$：元のデータ行列（400 × 4096）
- $W$：係数行列（各サンプルの成分の重み）
- $H$：基底行列（顔の「部品」）

**制約**：すべての成分が非負 → 加算のみで顔を表現

In [ ]:
# NMF の学習
n_components_nmf = 20

nmf = NMF(
    n_components=n_components_nmf,
    init='nndsvda',   # 非負初期化（収束が安定）
    max_iter=500,
    random_state=42
)
nmf.fit(X_train)

print(f'NMF 学習完了（n_components={n_components_nmf}）')
print(f'再構成誤差: {nmf.reconstruction_err_:.4f}')
print(f'基底行列の形状: {nmf.components_.shape}')
print(f'基底行列の値の範囲: [{nmf.components_.min():.4f}, {nmf.components_.max():.4f}]')
print(f'→ すべて非負 ✓')

### 3.1 PCA 固有顔 vs NMF 局所顔の比較

In [ ]:
# PCA（n=20）と NMF（n=20）の基底を比較
pca_20 = PCA(n_components=20, random_state=42).fit(X_train)

fig, axes = plt.subplots(4, 10, figsize=(18, 8))

# 上2行：PCA の固有顔
for i in range(20):
    row = i // 10
    col = i % 10
    ef = pca_20.components_[i].reshape(64, 64)
    vmax = np.abs(ef).max()
    axes[row, col].imshow(ef, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    if row == 0 and col == 0:
        axes[row, col].set_ylabel('PCA\n（固有顔）', fontsize=9, rotation=0,
                                   labelpad=45, va='center')
    axes[row, col].axis('off')

# 下2行：NMF の局所顔
for i in range(20):
    row = 2 + i // 10
    col = i % 10
    lf = nmf.components_[i].reshape(64, 64)
    axes[row, col].imshow(lf, cmap='hot')
    if row == 2 and col == 0:
        axes[row, col].set_ylabel('NMF\n（局所顔）', fontsize=9, rotation=0,
                                   labelpad=45, va='center')
    axes[row, col].axis('off')

plt.suptitle(
    'PCA 固有顔（上）vs NMF 局所顔（下）\n'
    'PCA：正負混在の幽霊的パターン  |  NMF：非負のみ・局所的な顔部品',
    fontsize=12
)
plt.tight_layout()
plt.show()

print('観察ポイント：')
print('  PCA固有顔：顔全体にわたるパターン（正の部分＋負の部分）')
print('  NMF局所顔：目・鼻・口・輪郭など局所的な部品に近い形')

### 3.2 NMF による顔の再構成

In [ ]:
# NMF で顔を再構成（加算のみ）
sample_idx = 5
sample = X_test[sample_idx]

# NMF 係数を求める（W の行に相当）
W_sample = nmf.transform(sample.reshape(1, -1))[0]  # (n_components,)

# 上位成分を1つずつ加算していく過程を可視化
sorted_idx = np.argsort(W_sample)[::-1]
n_steps = [1, 3, 5, 10, 15, 20]

fig, axes = plt.subplots(2, len(n_steps)+1, figsize=(16, 5))

# 元画像
axes[0, 0].imshow(sample.reshape(64,64), cmap='gray')
axes[0, 0].set_title('元画像', fontsize=9)
axes[0, 0].axis('off')
axes[1, 0].axis('off')

for ax_col, n in enumerate(n_steps, 1):
    # 上位n個の部品で再構成
    recon = np.zeros(4096)
    for idx in sorted_idx[:n]:
        recon += W_sample[idx] * nmf.components_[idx]
    axes[0, ax_col].imshow(recon.reshape(64,64), cmap='gray')
    axes[0, ax_col].set_title(f'部品 top-{n}', fontsize=9)
    axes[0, ax_col].axis('off')

    # 使われている部品（重みが大きいもの）
    for part_i in range(min(n, 4)):  # 上位4部品を表示
        pass
    axes[1, ax_col].axis('off')

plt.suptitle('NMF による顔の再構成：部品を加算していく過程', fontsize=11)
plt.tight_layout()
plt.show()

print('NMFの解釈：')
print('  各顔 = 部品1の重み × 部品1 + 部品2の重み × 部品2 + ...')
print('  すべて非負 → 「引き算せず足し合わせるだけ」で顔を表現')

### 3.3 NMF の係数空間での男女分布

In [ ]:
# NMF 係数の男女分布（上位2成分）
W_test  = nmf.transform(X_test)
W_train = nmf.transform(X_train)

# 男女の分離が最も明確な2成分を選ぶ
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
lda = LinearDiscriminantAnalysis(n_components=1).fit(W_train, ys_train)
lda_scores = lda.transform(W_test).ravel()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# 左：上位2NMF成分の散布図
colors = {0: 'steelblue', 1: 'tomato'}
for sex, label in zip([0,1],['男性','女性']):
    mask = ys_test == sex
    axes[0].scatter(W_test[mask,0], W_test[mask,1],
                    c=colors[sex], s=50, alpha=0.7, label=label)
axes[0].set_xlabel('NMF成分1の係数', fontsize=11)
axes[0].set_ylabel('NMF成分2の係数', fontsize=11)
axes[0].set_title('NMF係数空間での男女分布（上位2成分）', fontsize=11)
axes[0].legend()

# 右：LDA スコアの分布
for sex, label in zip([0,1],['男性','女性']):
    mask = ys_test == sex
    axes[1].hist(lda_scores[mask], bins=15, alpha=0.6,
                 color=colors[sex], label=label)
axes[1].set_xlabel('LDA スコア', fontsize=11)
axes[1].set_ylabel('度数', fontsize=11)
axes[1].set_title('NMF係数のLDA射影\n（男女の分離度の確認）', fontsize=11)
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 4. k-means クラスタリング

**教師なし学習の典型**：ラベルなしでデータをグループ化

本日は PCA 空間（100次元）で顔をクラスタリングし，
クラスター中心を顔画像として可視化します。

In [ ]:
# 第5回と同じ PCA 前処理
pca = PCA(n_components=100, whiten=True, random_state=42)
X_pca_train = pca.fit_transform(X_train)
X_pca_all   = pca.transform(X)  # 全データ（クラスタリング用）
print('PCA変換完了')
print(f'形状: {X_pca_all.shape}')

### 4.1 最適 k の選択：エルボー法とシルエット係数

In [ ]:
# k を変えてエルボー法とシルエット係数を比較
k_range   = range(2, 21)
inertias  = []  # Within-cluster sum of squares
silhouettes = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_pca_all)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_pca_all, labels, sample_size=200)
    silhouettes.append(sil)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# エルボー法
axes[0].plot(list(k_range), inertias, 'steelblue', marker='o', linewidth=2)
axes[0].set_xlabel('クラスター数 k', fontsize=11)
axes[0].set_ylabel('Within-SS（惰性）', fontsize=11)
axes[0].set_title('エルボー法\n「肘」の位置が適切な k', fontsize=11)

# シルエット係数
best_k = list(k_range)[np.argmax(silhouettes)]
axes[1].plot(list(k_range), silhouettes, 'tomato', marker='s', linewidth=2)
axes[1].axvline(best_k, color='gray', linestyle='--', linewidth=1.5,
                label=f'最良 k={best_k}')
axes[1].set_xlabel('クラスター数 k', fontsize=11)
axes[1].set_ylabel('シルエット係数', fontsize=11)
axes[1].set_title('シルエット係数（高いほど良い）', fontsize=11)
axes[1].legend()

plt.tight_layout()
plt.show()
print(f'シルエット係数が最大の k: {best_k}')

### 4.2 クラスター中心を顔画像として可視化

In [ ]:
# k=10 でクラスタリング（見やすい枚数）
K = 10
km = KMeans(n_clusters=K, random_state=42, n_init=10)
cluster_labels = km.fit_predict(X_pca_all)

# クラスター中心を元の画像空間に逆変換
centers_img = pca.inverse_transform(km.cluster_centers_)  # (K, 4096)

fig, axes = plt.subplots(2, K, figsize=(18, 5))

for k in range(K):
    # 上段：クラスター中心（平均顔）
    axes[0, k].imshow(centers_img[k].reshape(64,64), cmap='gray')
    n_members = (cluster_labels == k).sum()
    n_female  = ((cluster_labels == k) & (y_sex == 1)).sum()
    axes[0, k].set_title(f'C{k}\nn={n_members}\n女性:{n_female}',
                          fontsize=8)
    axes[0, k].axis('off')

    # 下段：各クラスターの代表サンプル
    cluster_members = np.where(cluster_labels == k)[0]
    if len(cluster_members) > 0:
        rep_idx = cluster_members[0]
        axes[1, k].imshow(X[rep_idx].reshape(64,64), cmap='gray')
        axes[1, k].set_title('代表例', fontsize=8)
        axes[1, k].axis('off')

plt.suptitle(
    f'k-means (k={K})：上段=クラスター中心（仮想的な「平均顔」），下段=代表サンプル',
    fontsize=11
)
plt.tight_layout()
plt.show()

In [ ]:
# クラスタリングの評価（正解ラベルとの照合）
# k-meansは教師なしなので厳密な評価ではないが，傾向を確認
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# 人物IDとの一致度（どれだけ同じ人がクラスタリングされるか）
ari_person = adjusted_rand_score(y, cluster_labels)
nmi_person = normalized_mutual_info_score(y, cluster_labels)

# 男女ラベルとの一致度
ari_sex = adjusted_rand_score(y_sex, cluster_labels)

print(f'=== k-means (k={K}) の評価 ===')
print(f'人物ID との ARI : {ari_person:.4f}  （1.0 = 完全一致）')
print(f'人物ID との NMI : {nmi_person:.4f}')
print(f'男女ラベル との ARI: {ari_sex:.4f}')
print()
print('解釈：')
print('  ARI > 0 = ランダムより有意にグループ化できている')
print('  教師なし学習でも顔の特徴を捉えたグループが形成される')

### 4.3 PCA 2次元空間でのクラスター可視化

In [ ]:
# 2次元 PCA でクラスターを可視化
pca2 = PCA(n_components=2, random_state=42)
X_2d = pca2.fit_transform(X)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cmap_k = plt.get_cmap('tab10')

# 左：k-meansクラスター
for k in range(K):
    mask = cluster_labels == k
    axes[0].scatter(X_2d[mask,0], X_2d[mask,1],
                    s=30, alpha=0.7, color=cmap_k(k/K), label=f'C{k}')
# クラスター中心
centers_2d = pca2.transform(pca.inverse_transform(km.cluster_centers_))
axes[0].scatter(centers_2d[:,0], centers_2d[:,1],
                s=200, marker='*', color='black', zorder=5, label='中心')
axes[0].set_title(f'k-means (k={K}) クラスター\nPCA 2次元空間', fontsize=11)
axes[0].legend(bbox_to_anchor=(1.02,1), loc='upper left', fontsize=8)

# 右：男女ラベルで色分け
for sex, label, color in zip([0,1],['男性','女性'],['steelblue','tomato']):
    mask = y_sex == sex
    axes[1].scatter(X_2d[mask,0], X_2d[mask,1],
                    s=30, alpha=0.7, color=color, label=label)
axes[1].set_title('男女ラベルでの分布\nPCA 2次元空間', fontsize=11)
axes[1].legend()

for ax in axes:
    ax.set_xlabel('第1主成分', fontsize=10)
    ax.set_ylabel('第2主成分', fontsize=10)

plt.tight_layout()
plt.show()

---
## 5. SOM（自己組織化マップ）概観

**本セクションは位置づけの理解が目的です。実装の詳細には立ち入りません。**

In [ ]:
try:
    from minisom import MiniSom

    # 小さな SOM でデモ（計算時間を抑えるため小サイズ）
    som_size = 8  # 8×8 グリッド
    som = MiniSom(
        x=som_size, y=som_size,
        input_len=X_pca_all.shape[1],
        sigma=1.5, learning_rate=0.5,
        random_seed=42
    )
    som.random_weights_init(X_pca_all)
    som.train(X_pca_all, num_iteration=500, verbose=False)

    # 各サンプルの BMU（Best Matching Unit）を取得
    bmu_positions = np.array([som.winner(x) for x in X_pca_all])

    # SOM グリッド上での男女分布を可視化
    grid_sex = np.full((som_size, som_size), np.nan)
    for i, (bmu_x, bmu_y) in enumerate(bmu_positions):
        if np.isnan(grid_sex[bmu_x, bmu_y]):
            grid_sex[bmu_x, bmu_y] = 0
        grid_sex[bmu_x, bmu_y] += y_sex[i]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Uマトリクス（隣接ユニット間の距離）
    u_matrix = som.distance_map()
    im0 = axes[0].imshow(u_matrix, cmap='bone_r')
    plt.colorbar(im0, ax=axes[0])
    # 各サンプルをマップ上にプロット
    markers = {0: 'o', 1: '*'}
    colors_som = {0: 'steelblue', 1: 'tomato'}
    for i, (bmu_x, bmu_y) in enumerate(bmu_positions):
        axes[0].plot(
            bmu_y + 0.5 + np.random.uniform(-0.3, 0.3),
            bmu_x + 0.5 + np.random.uniform(-0.3, 0.3),
            markers[y_sex[i]], color=colors_som[y_sex[i]],
            markersize=5, alpha=0.6
        )
    axes[0].set_title(f'SOM Uマトリクス ({som_size}×{som_size})\n明るい領域＝クラスター境界', fontsize=11)

    # 距離マップ上での男女ヒートマップ
    axes[1].imshow(u_matrix, cmap='bone_r', alpha=0.5)
    im1 = axes[1].imshow(np.nan_to_num(grid_sex), cmap='RdBu_r',
                          alpha=0.7, vmin=0)
    plt.colorbar(im1, ax=axes[1], label='女性の数')
    axes[1].set_title('SOMマップ上での男女分布\n赤=女性が多い領域', fontsize=11)

    plt.suptitle('SOM（自己組織化マップ）：高次元データを2次元に写像\n'
                 '似た顔は近くに，異なる顔は遠くに配置される', fontsize=11)
    plt.tight_layout()
    plt.show()
    print('SOM のポイント：')
    print('  k-means との違い：トポロジー（近さの関係）を保存')
    print('  脳の皮質マップとの類似：Hubel & Wiesel（第7回）へつながる')

except ImportError:
    print('minisom がインストールできない場合は以下の概念図を参照してください')
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.set_xlim(0, 10); ax.set_ylim(0, 10)
    np.random.seed(42)
    n_points = 80
    pts = np.random.randn(n_points, 2) * 2 + 5
    colors_pts = ['steelblue'] * 70 + ['tomato'] * 10
    ax.scatter(pts[:,0], pts[:,1], c=colors_pts, s=60, alpha=0.7)
    for i in range(0, 10, 2):
        for j in range(0, 10, 2):
            ax.add_patch(plt.Rectangle((i, j), 2, 2, fill=False,
                         edgecolor='gray', linewidth=0.5, alpha=0.5))
    ax.set_title('SOM 概念図：グリッドユニットが高次元データを2Dに写像\n'
                 '青=男性，赤=女性，同じユニットに写像されたものが「似ている」', fontsize=11)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

### 5.1 Infomax の位置づけ（概説）

**情報量最大化の原理（Bell & Sejnowski, 1995）**

$$\max_W H(y) = \max_W H(f(Wx))$$

| 手法 | 最適化の目標 | 得られるもの |
|------|-----------|------------|
| PCA | 分散最大化 | 直交成分（無相関） |
| NMF | 再構成誤差最小化（非負制約） | 局所的な部品 |
| ICA（Infomax） | 独立性最大化 | 統計的独立成分 |

→ 独立成分分析（ICA）の理論的基盤  
→ 深入りはしないが，「情報量」が最適化の目標になることを押さえる  
→ 第4回の**交差エントロピー = 情報量の差**との接続

---
## 6. TF-IDF / Naive Bayes

ここでのみ扱うトピックの概説です。
実習は簡単なデモのみ。

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.datasets import fetch_20newsgroups
from sklearn.metrics import accuracy_score

# 20 Newsgroups から2カテゴリを選択（計算を速くするため）
categories = ['sci.med', 'comp.graphics']
news_train = fetch_20newsgroups(subset='train', categories=categories,
                                remove=('headers','footers','quotes'))
news_test  = fetch_20newsgroups(subset='test',  categories=categories,
                                remove=('headers','footers','quotes'))

print('=== 20 Newsgroups データセット（2カテゴリ）===')
print(f'  カテゴリ   : {categories}')
print(f'  訓練サンプル: {len(news_train.data)}')
print(f'  テストサンプル: {len(news_test.data)}')
print(f'\n  サンプル文書（先頭200字）:\n  "{news_train.data[0][:200]}..."')

=== 20 Newsgroups データセット（2カテゴリ）===
  カテゴリ   : ['sci.med', 'comp.graphics']
  訓練サンプル: 1178
  テストサンプル: 785

  サンプル文書（先頭200字）:
  "
	It depends on what kind of the polygons. 
	Convex - simple, concave - trouble, concave with loop(s)
	inside - big trouble.

	Of cause, you can use the box test to avoid checking
	each edges. Accordi..."


In [22]:
# TF-IDF ベクトル化
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_tr_tfidf = tfidf.fit_transform(news_train.data)
X_te_tfidf = tfidf.transform(news_test.data)

print('=== TF-IDF ベクトル化 ===')
print(f'  文書×単語 行列の形状: {X_tr_tfidf.shape}')
print(f'  (疎行列: 大部分がゼロ)')

# カテゴリ別の重要単語を表示
feature_names = tfidf.get_feature_names_out()
for cat_id, cat_name in enumerate(categories):
    mask = news_train.target == cat_id
    mean_tfidf = X_tr_tfidf[mask].mean(axis=0).A1
    top5_idx = mean_tfidf.argsort()[-5:][::-1]
    top5_words = [feature_names[i] for i in top5_idx]
    print(f'\n  {cat_name} の重要単語（TF-IDF上位5）: {top5_words}')

=== TF-IDF ベクトル化 ===
  文書×単語 行列の形状: (1178, 5000)
  (疎行列: 大部分がゼロ)

  sci.med の重要単語（TF-IDF上位5）: ['graphics', 'thanks', 'files', 'image', 'know']

  comp.graphics の重要単語（TF-IDF上位5）: ['msg', 'edu', 'don', 'know', 'people']


In [ ]:
# Naive Bayes 分類器
nb = MultinomialNB(alpha=0.1)
nb.fit(X_tr_tfidf, news_train.target)
y_pred_nb = nb.predict(X_te_tfidf)

print('=== Naive Bayes 分類結果 ===')
print(f'  テスト精度: {accuracy_score(news_test.target, y_pred_nb)*100:.1f}%')
print()

# Naive Bayes の原理を確認
print('=== Naive Bayes の原理 ===')
print('  P(y | x) ∝ P(y) × Π P(xⱼ | y)')
print()
print('  第4回との接続：')
print('    P(x|y) = クラス条件付き尤度')
print('    P(y) = 事前確率（訓練データのクラス比率）')
print('    → ベイズの定理 + 最尤推定')
print()

# ロジスティック回帰との比較
from sklearn.linear_model import LogisticRegression
lr_text = LogisticRegression(max_iter=1000).fit(X_tr_tfidf, news_train.target)
y_pred_lr = lr_text.predict(X_te_tfidf)

print('=== 比較 ===')
print(f'  Naive Bayes       : {accuracy_score(news_test.target, y_pred_nb)*100:.1f}%')
print(f'  ロジスティック回帰  : {accuracy_score(news_test.target, y_pred_lr)*100:.1f}%')
print()
print('Naive Bayes の特徴：')
print('  ✅ 少ないデータでも動く（生成モデル）')
print('  ✅ 計算が高速')
print('  ⚠️  特徴量の独立性の仮定が強すぎる（テキストでは）')

---
## 7. まとめ

In [ ]:
print('='*65)
print('  第6回まとめ：教師なし学習と次元削減')
print('='*65)
print("""
【PCA の理論（第5回で「使った」→ 今日「理解した」）】
  - 最大分散方向への射影 = 情報量の最大保存
  - 固有ベクトル = 顔空間の基底（固有顔）
  - SVD = PCA の数値安定な実装

【NMF（非負値行列因子分解）】
  - V ≈ W × H,  W,H ≥ 0
  - 局所的な「顔部品」表現（PCA との対比）
  - 足し合わせるだけで顔を再構成

【k-means クラスタリング】
  - Within-SS 最小化，エルボー法・シルエット係数で k を選択
  - クラスター中心を逆変換 → 仮想的な「平均顔」

【SOM・Infomax（位置づけのみ）】
  - SOM：トポロジー保存マップ（脳の皮質マップとの類似）
  - Infomax：情報量最大化 → ICA の理論的基盤

【TF-IDF / Naive Bayes（概説）】
  - TF-IDF：文書の数値表現（重要単語を重視）
  - Naive Bayes：生成モデル（第4回の尤度と直結）

【次回（第7回）】
  前半の総括 + ニューラルネットワーク導入
  Hubel & Wiesel，パーセプトロン，誤差逆伝播
""")

---
## 準備学習課題

**課題1**
ニューラルネットワークが「特徴量設計」をどのように置き換えるかを説明せよ。  
（ヒント：第5・6回で行った PCA・NMF という手動の特徴量設計と比較する）

---
### 解答欄

**課題1**:


---
## 参考文献

1. Lee, D. D., & Seung, H. S. (1999). Learning the parts of objects by non-negative matrix factorization. *Nature, 401*, 788–791.
2. Kohonen, T. (1982). Self-organized formation of topologically correct feature maps. *Biological Cybernetics, 43*(1), 59–69.
3. Bell, A. J., & Sejnowski, T. J. (1995). An information-maximization approach to blind separation and blind deconvolution. *Neural Computation, 7*(6), 1129–1159.
4. Freund, Y., & Schapire, R. E. (1997). A decision-theoretic generalization. *JCSS, 55*(1), 119–139.
5. sklearn Faces recognition example: https://scikit-learn.org/stable/auto_examples/applications/plot_face_recognition.html

---
*人工知能I 第6回実習ノートブック | 担当：浅川伸一*